# Cycling Optimization Notebook

**Author:** Vanessa Fioravanti (personal notebook template)

**Purpose:** This notebook parses ride data (FIT/CSV), computes training zones, evaluates training efficiency, runs Monte Carlo FTP simulations, and solves a TSS Budget optimization problem using Pyomo. It contains placeholders so you can drop your own files and run the analyses in SageMaker.

---

**How to use:**

1. Upload your FIT or CSV ride files into the indicated folder (placeholder paths).
2. Install required Python packages in your SageMaker environment if missing.
3. Run cells sequentially.


In [ ]:
# 0. Environment setup and imports
import os
import sys
import math
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

try:
    import fitparse
    FITPARSE_AVAILABLE = True
except Exception:
    FITPARSE_AVAILABLE = False

try:
    import pyomo.environ as pyo
    PYOMO_AVAILABLE = True
except Exception:
    PYOMO_AVAILABLE = False

print('fitparse available:', FITPARSE_AVAILABLE)
print('pyomo available:', PYOMO_AVAILABLE)
print('numpy/pandas/matplotlib versions:', np.__version__, pd.__version__)


## 1. File placeholders

Set the paths where your ride files are stored. Edit the `DATA_DIR` variable.

In [ ]:
DATA_DIR = '/path/to/your/fit_or_csv_folder'  # <<-- change this
OUTPUT_DIR = '/mnt/data/cycling_results'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('DATA_DIR set to', DATA_DIR)

### 1.2 FIT parsing helper (if using .fit files)

If you have `.fit` files, the function below attempts to parse them using `fitparse`. If `fitparse` is not available, you can convert your FIT files to CSV using Garmin Connect or other tools and use the CSV loader instead.

In [ ]:
from datetime import datetime, timezone

def parse_fit_power(fitfile_path):
    if not FITPARSE_AVAILABLE:
        raise RuntimeError('fitparse not installed. Convert FIT to CSV or install fitparse.')
    from fitparse import FitFile
    fitfile = FitFile(fitfile_path)
    records = []
    for record in fitfile.get_messages('record'):
        rec = {}
        for d in record:
            rec[d.name] = d.value
        records.append(rec)
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    if 'power' not in df.columns:
        df['power'] = np.nan
    if 'heart_rate' not in df.columns:
        df['heart_rate'] = np.nan
    keep = [c for c in ['timestamp','power','heart_rate','cadence','speed'] if c in df.columns]
    return df[keep].copy()

print('Helper ready. Fitparse available:', FITPARSE_AVAILABLE)


### 1.3 CSV loader (if using pre-exported CSV files)

CSV files are assumed to contain a timestamp column and a `power` column. Modify if needed.

In [ ]:
def load_csv_power(csv_path, timestamp_col=None):
    df = pd.read_csv(csv_path)
    if timestamp_col is None:
        candidates = [c for c in df.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
        timestamp_col = candidates[0] if candidates else None
    if timestamp_col:
        df['timestamp'] = pd.to_datetime(df[timestamp_col])
    else:
        df['timestamp'] = pd.to_datetime(df.index, unit='s', origin='unix')
    power_candidates = [c for c in df.columns if 'power' in c.lower()]
    if not power_candidates:
        raise RuntimeError(f'No power column found in {csv_path}. Columns: {df.columns.tolist()}')
    df['power'] = df[power_candidates[0]]
    hr_candidates = [c for c in df.columns if 'hr' in c.lower() or 'heart' in c.lower()]
    if hr_candidates:
        df['heart_rate'] = df[hr_candidates[0]]
    else:
        df['heart_rate'] = np.nan
    return df[['timestamp','power','heart_rate']].copy()

print('CSV loader ready')


### 1.4 Auto-detect files and load a few rides (placeholder behavior)

In [ ]:
def load_rides(data_dir, max_files=50):
    data_dir = Path(data_dir)
    rides = []
    fit_files = sorted(list(data_dir.glob('*.fit')))
    csv_files = sorted(list(data_dir.glob('*.csv')))
    if fit_files:
        print('Found', len(fit_files), '.fit files. Parsing up to', max_files)
        for p in fit_files[:max_files]:
            try:
                df = parse_fit_power(p)
                df['file'] = str(p.name)
                rides.append(df)
            except Exception as e:
                print('Error parsing', p, e)
    elif csv_files:
        print('Found', len(csv_files), '.csv files. Loading up to', max_files)
        for p in csv_files[:max_files]:
            try:
                df = load_csv_power(p)
                df['file'] = str(p.name)
                rides.append(df)
            except Exception as e:
                print('Error loading', p, e)
    else:
        print('No files found in', data_dir, '. Generating synthetic dataset for demo.')
        rng = np.random.default_rng(42)
        def synthetic_ride(duration_min=60, mean_power=150, var=30):
            n = duration_min*4
            t0 = pd.Timestamp('2025-01-01')
            ts = pd.date_range(t0, periods=n, freq='15s')
            power = np.clip(rng.normal(mean_power, var, size=n).astype(float), 0, 800)
            hr = np.clip(120 + (power-100)/2 + rng.normal(0,5,size=n), 50, 200)
            return pd.DataFrame({'timestamp': ts, 'power': power, 'heart_rate': hr})
        rides = []
        rides.append(synthetic_ride(70, mean_power=320).assign(file='tuesday_vo2.fit'))
        rides.append(synthetic_ride(90, mean_power=140).assign(file='thursday_endurance.fit'))
        rides.append(synthetic_ride(220, mean_power=155).assign(file='saturday_long.fit'))
    return rides

rides = load_rides(DATA_DIR)
print('Loaded', len(rides), 'rides (showing filenames):', [r.get('file', 'synthetic') for r in rides])


## 2. Compute Training Zones (Idea 1.1)

We use a standard 6-zone model based on your FTP. Edit the `FTP` variable if needed.

In [ ]:
FTP = 190.0  # change if needed
zones = {
    'Z1': (0, 0.55*FTP),
    'Z2': (0.55*FTP, 0.75*FTP),
    'Z3': (0.75*FTP, 0.9*FTP),
    'Z4': (0.9*FTP, 1.05*FTP),
    'Z5': (1.05*FTP, 1.2*FTP),
    'Z6': (1.2*FTP, 10*FTP),
}

def assign_zone(power):
    for z,(lo,hi) in zones.items():
        if lo <= power < hi:
            return z
    return 'Z6'

processed = []
for df in rides:
    d = df.copy()
    d = d.dropna(subset=['power']).reset_index(drop=True)
    if 'timestamp' not in d.columns:
        d['timestamp'] = pd.date_range(pd.Timestamp.now(), periods=len(d), freq='15s')
    d['dt'] = d['timestamp'].diff().dt.total_seconds().fillna(15)
    d['zone'] = d['power'].apply(assign_zone)
    d['duration_h'] = d['dt'] / 3600.0
    d['np_proxy'] = d['power']
    d['tss_proxy'] = ((d['np_proxy']/FTP)**2) * d['duration_h'] * 100
    processed.append(d)

zone_summary = []
for d in processed:
    fname = d['file'].iloc[0] if 'file' in d.columns else 'ride'
    zs = d.groupby('zone')['dt'].sum().rename('seconds').reset_index()
    zs['hours'] = zs['seconds']/3600.0
    zs['file'] = fname
    zone_summary.append(zs)
zone_summary_df = pd.concat(zone_summary, ignore_index=True)
zone_summary_pivot = zone_summary_df.pivot_table(index='file', columns='zone', values='hours', fill_value=0)
zone_summary_pivot['total_hours'] = zone_summary_pivot.sum(axis=1)
zone_summary_pivot = zone_summary_pivot.sort_values('total_hours', ascending=False)
zone_summary_pivot


In [ ]:
zone_summary_pivot.plot(kind='bar', stacked=True, figsize=(12,5))
plt.title('Time in zones per ride (hours)')
plt.ylabel('Hours')
plt.xlabel('Ride file')
plt.tight_layout()
plt.show()

## 3. Training efficiency metrics (Idea 2.D)

We compute simple efficiency metrics per ride: average power, NP proxy, HR drift, and an efficiency metric.

In [ ]:
ride_metrics = []
for d in processed:
    fname = d['file'].iloc[0] if 'file' in d.columns else 'ride'
    total_tss = d['tss_proxy'].sum()
    avg_power = d['power'].mean()
    np_proxy = d['np_proxy'].mean()
    hr_start = d['heart_rate'].iloc[0] if 'heart_rate' in d.columns else np.nan
    hr_end = d['heart_rate'].iloc[-1] if 'heart_rate' in d.columns else np.nan
    hr_drift = hr_end - hr_start if not np.isnan(hr_start) and not np.isnan(hr_end) else np.nan
    duration_h = d['duration_h'].sum()
    delta_perf_est = (np_proxy - 150) / 100.0
    eff = delta_perf_est / (total_tss+1e-6)
    ride_metrics.append({
        'file': fname,
        'total_tss': total_tss,
        'avg_power': avg_power,
        'np_proxy': np_proxy,
        'hr_drift': hr_drift,
        'duration_h': duration_h,
        'delta_perf_est': delta_perf_est,
        'efficiency': eff
    })

ride_metrics_df = pd.DataFrame(ride_metrics).set_index('file')
ride_metrics_df


In [ ]:
ride_metrics_df[['total_tss','delta_perf_est','efficiency']].plot(kind='bar', subplots=True, figsize=(12,8))
plt.suptitle('Ride-level metrics: TSS, est. delta performance and efficiency')
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

## 4. Monte Carlo FTP simulations (Idea 2.C)

A simple stochastic model for FTP response to weekly TSS.

In [ ]:
import random

def simulate_ftp(initial_ftp, weekly_tss_series, alpha=0.0025, beta=0.001, noise_sd=0.5, weeks=12, runs=200):
    results = np.zeros((runs, weeks+1))
    for r in range(runs):
        ftp = initial_ftp
        results[r,0] = ftp
        for w in range(1,weeks+1):
            weekly_tss = weekly_tss_series[min(w-1, len(weekly_tss_series)-1)]
            fatigue = weekly_tss * 0.1
            noise = np.random.normal(0, noise_sd)
            ftp = ftp + alpha * weekly_tss - beta * fatigue + noise
            results[r,w] = ftp
    return results

weekly_tss_baseline = sum([d['tss_proxy'].sum() for d in processed])
print('Estimated baseline weekly TSS (proxy):', weekly_tss_baseline)

scenarios = {
    'baseline': [weekly_tss_baseline]*12,
    'plus5pc': [weekly_tss_baseline*1.05]*12,
    'plus10pc': [weekly_tss_baseline*1.10]*12,
    'plus20pc': [weekly_tss_baseline*1.20]*12,
    'plus10_vo2_only': [weekly_tss_baseline + (processed[0]['tss_proxy'].sum()*0.10)]*12 if len(processed)>0 else [weekly_tss_baseline]*12
}

sim_results = {k: simulate_ftp(FTP, v, runs=300) for k,v in scenarios.items()}
print('Simulations finished for scenarios:', list(sim_results.keys()))


In [ ]:
def plot_simulation(sim_mat, label):
    weeks = sim_mat.shape[1]-1
    mean = sim_mat.mean(axis=0)
    p10 = np.percentile(sim_mat,10,axis=0)
    p90 = np.percentile(sim_mat,90,axis=0)
    x = np.arange(0, weeks+1)
    plt.plot(x, mean, label=label)
    plt.fill_between(x, p10, p90, alpha=0.2)

plt.figure(figsize=(10,6))
for k,v in sim_results.items():
    plot_simulation(v,k)
plt.xlabel('Week')
plt.ylabel('FTP (W)')
plt.title('Monte Carlo FTP simulation: scenarios compared')
plt.legend()
plt.grid(True)
plt.show()


## 5. TSS Budget Optimization with Pyomo (Idea 1.3)

This section sets up a small optimization problem using Pyomo. If Pyomo is not installed, install it and a solver (e.g., glpk).

In [ ]:
# 5.1 Build data for optimization
session_template = [
    {'name':'VO2', 'duration_h': 70/60.0, 'baseline_tss': processed[0]['tss_proxy'].sum() if len(processed)>0 else 50},
    {'name':'Endurance', 'duration_h': 90/60.0, 'baseline_tss': processed[1]['tss_proxy'].sum() if len(processed)>1 else 60},
    {'name':'Long', 'duration_h': 220/60.0, 'baseline_tss': processed[2]['tss_proxy'].sum() if len(processed)>2 else 120},
]
session_df = pd.DataFrame(session_template).set_index('name')
session_df


In [ ]:
if not PYOMO_AVAILABLE:
    print('Pyomo not available. Please install pyomo in your SageMaker kernel (pip install pyomo) and ensure a solver is available (e.g., glpk).')
else:
    model = pyo.ConcreteModel()
    S = list(session_df.index)
    model.S = pyo.Set(initialize=S)
    model.x = pyo.Var(model.S, domain=pyo.NonNegativeReals, bounds=(0.5,2.0))
    baseline_tss = {s: float(session_df.loc[s,'baseline_tss']) for s in S}
    duration_h = {s: float(session_df.loc[s,'duration_h']) for s in S}
    def tss_expr(model,s):
        return baseline_tss[s] * model.x[s]
    model.tss = pyo.Expression(model.S, rule=tss_expr)
    a_param = {s: 1.0 for s in S}
    def gain_expr(model,s):
        return a_param[s] * pyo.sqrt(model.tss[s])
    model.gain = pyo.Expression(model.S, rule=gain_expr)
    model.obj = pyo.Objective(expr = sum(model.gain[s] for s in model.S), sense=pyo.maximize)
    WEEKLY_TSS_BUDGET = max(sum(baseline_tss.values())*1.10, 200)
    WEEKLY_HOURS_BUDGET = 6.0
    model.tss_budget = pyo.Constraint(expr = sum(model.tss[s] for s in model.S) <= WEEKLY_TSS_BUDGET)
    model.time_budget = pyo.Constraint(expr = sum(duration_h[s] * model.x[s] for s in model.S) <= WEEKLY_HOURS_BUDGET)
    try:
        solver = pyo.SolverFactory('glpk')
        result = solver.solve(model, tee=False)
        print('Solver status:', result.solver.status, 'Termination condition:', result.solver.termination_condition)
        optimized = {s: pyo.value(model.x[s]) for s in model.S}
        optimized_tss = {s: float(pyo.value(model.tss[s])) for s in model.S}
        optimized_gain = {s: float(pyo.value(model.gain[s])) for s in model.S}
        print('Optimized multipliers:', optimized)
    except Exception as e:
        print('Solver failed or not available in this environment:', e)


In [ ]:
if PYOMO_AVAILABLE:
    baseline = {s: float(session_df.loc[s,'baseline_tss']) for s in session_df.index}
    opt_x = optimized if 'optimized' in globals() else {s:1.0 for s in session_df.index}
    comp = pd.DataFrame({
        'baseline_tss': [baseline[s] for s in session_df.index],
        'opt_multiplier': [opt_x[s] for s in session_df.index],
        'opt_tss': [optimized_tss[s] for s in session_df.index] if 'optimized_tss' in globals() else [baseline[s] for s in session_df.index],
        'baseline_duration_h': [float(session_df.loc[s,'duration_h']) for s in session_df.index],
        'opt_duration_h': [float(session_df.loc[s,'duration_h'])*opt_x[s] for s in session_df.index]
    }, index=session_df.index)
    comp['opt_gain'] = [optimized_gain[s] for s in session_df.index] if 'optimized_gain' in globals() else np.nan
    comp


## 6. Summary tables & export
Save computed summaries and figures to the OUTPUT_DIR for use in your Medium post.

In [ ]:
zone_summary_pivot.to_csv(os.path.join(OUTPUT_DIR,'zone_summary_per_ride.csv'))
ride_metrics_df.to_csv(os.path.join(OUTPUT_DIR,'ride_metrics_summary.csv'))
if 'comp' in globals():
    comp.to_csv(os.path.join(OUTPUT_DIR,'optimization_comparison.csv'))
print('Exported summary CSVs to', OUTPUT_DIR)

figpath = os.path.join(OUTPUT_DIR,'monte_carlo_ftp.png')
plt.figure(figsize=(10,6))
for k,v in sim_results.items():
    plot_simulation(v,k)
plt.xlabel('Week')
plt.ylabel('FTP (W)')
plt.title('Monte Carlo FTP simulation: scenarios compared')
plt.legend()
plt.grid(True)
plt.savefig(figpath, bbox_inches='tight', dpi=150)
plt.close()
print('Saved Monte Carlo plot to', figpath)


## 7. Next steps and customization ideas

- Replace proxies with accurate Normalized Power and official TSS.
- Fit simulation parameters using historical data.
- Tune optimization gain surrogates using regression results.
